[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Реализуйте **Rotary Position Embedding (RoPE)** — позиционное кодирование, которое поворачивает пары координат Q и K на угол, зависящий от позиции. В отличие от прибавления отдельного positional vector, RoPE встраивает относительное смещение непосредственно в dot-product attention.

## Поворот пары координат

Для пары $(x_{2i},x_{2i+1})$ на позиции $m$ задаётся частота $\theta_i=10000^{-2i/D}$ и угол $m\theta_i$. Поворот имеет вид:

$$\begin{pmatrix}x'_{2i}\\x'_{2i+1}\end{pmatrix}=\begin{pmatrix}\cos(m\theta_i)&-\sin(m\theta_i)\\\sin(m\theta_i)&\cos(m\theta_i)\end{pmatrix}\begin{pmatrix}x_{2i}\\x_{2i+1}\end{pmatrix}.$$

То есть $x'_{2i}=x_{2i}\cos(m\theta_i)-x_{2i+1}\sin(m\theta_i)$ и $x'_{2i+1}=x_{2i}\sin(m\theta_i)+x_{2i+1}\cos(m\theta_i)$. Каждая пара использует свою частоту: ранние пары вращаются быстрее, поздние — медленнее. Так кодируются разные масштабы расстояний.

Матрица поворота ортогональна, поэтому сохраняет норму каждой пары и всего вектора. Позиция 0 имеет нулевой угол: её преобразование является identity. Контракт требует чётный `D`, чтобы все координаты образовали пары.

## Свойство относительной позиции

Пусть $R_m$ и $R_n$ — блочно-диагональные повороты Q на позиции $m$ и K на позиции $n$. Тогда

$$(R_m q)^T(R_n k)=q^T R_m^T R_n k=q^T R_{n-m}k.$$

Следовательно, attention score зависит от относительного смещения $m-n$ (знак в записи меняется с соглашением о направлении вращения), а не от абсолютных позиций по отдельности. Если одновременно сдвинуть Q и K на одинаковое число позиций, их dot product не изменится.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

## Поток форм и план реализации

1. Для `q,k: (B,S,D)` постройте позиции `(S,)` и inverse frequencies `(D/2,)` на тех же device и dtype.
2. Их внешнее произведение даёт углы `(S,D/2)`; добавьте batch-ось для broadcasting.
3. Выделите чётные и нечётные координаты Q/K, примените одну и ту же таблицу sin/cos.
4. Соберите пары обратно в исходном порядке и сохраните форму `(B,S,D)`. Не отсоединяйте вычисления от autograd.

## Быстрые самопроверки

- Нормы Q/K до и после поворота совпадают с численным допуском.
- На позиции 0 rotated vector равен исходному.
- Совместный сдвиг двух позиций сохраняет их attention score.
- Формы, dtype и device не меняются; backward заполняет градиенты q и k.

## Частые ошибки

- Применить одинаковую частоту ко всем парам координат.
- Потерять множитель позиции или показатель `2i/D`.
- Сначала склеить все чётные, затем все нечётные координаты вместо восстановления пар.
- Повернуть Q и K разными таблицами углов или нарушить знак синуса.

## Материалы для глубокого изучения

- [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864) — раздел 3.2, особенно §3.2.2 о зависимости inner product от относительной позиции.
- [Hugging Face LLM course: Position embeddings](https://huggingface.co/learn/llm-course/chapter1/6) — место позиционной информации в Transformer.
- [PyTorch complex numbers](https://docs.pytorch.org/docs/stable/complex_numbers.html) — альтернативное представление пар как комплексных чисел и вращения как умножения на фазу.

## Вопросы для самопроверки

1. Почему RoPE сохраняет норму вектора?
2. Как из $R_m^TR_n$ возникает только разность позиций?
3. Зачем разным парам координат нужны разные частоты?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    pass

In [ ]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
result = apply_rope(q, k)
if result is None:
    print('Implement apply_rope, then rerun this cell.')
else:
    qr, kr = result
    print('Shape preserved:', qr.shape == q.shape)
    print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('rope')